# 16モデル × 16モデル 対戦

2つのフォルダにある `cluster_00` ～ `cluster_15` を比較します。

`run_matches_round_robin.py` には、学習時の `run_parallel_games()` と同じく
`--agent 名前=main.py` を直接渡します。

- `tournament.json` 不要
- schema解析不要
- ダミーrunner不要
- group内対戦は0試合
- group A × group B の `16 × 16 = 256` 組だけ対戦


In [4]:
from __future__ import annotations

import itertools
import json
import os
from pathlib import Path
import shlex
import subprocess
import sys

import pandas as pd
from IPython.display import display


def find_repo_root() -> Path:
    current = Path.cwd().resolve()

    for candidate in (current, *current.parents):
        if (
            candidate
            / "tools"
            / "run_matches_round_robin.py"
        ).is_file():
            return candidate

    raise FileNotFoundError(
        "tools/run_matches_round_robin.py がある"
        "リポジトリルートを見つけられません。"
    )


ROOT = find_repo_root()
RUNNER = (
    ROOT
    / "tools"
    / "run_matches_round_robin.py"
)
RESULTS_DIR = ROOT / "results"

venv_python = (
    ROOT
    / ".venv"
    / (
        "Scripts/python.exe"
        if os.name == "nt"
        else "bin/python"
    )
)

PYTHON = (
    venv_python
    if venv_python.is_file()
    else Path(sys.executable)
)

print(f"ROOT={ROOT}")
print(f"RUNNER={RUNNER}")
print(f"PYTHON={PYTHON}")


ROOT=/Users/naoki/Desktop/develop/pokemon_tgc_agent/pokemon-tcg-agent
RUNNER=/Users/naoki/Desktop/develop/pokemon_tgc_agent/pokemon-tcg-agent/tools/run_matches_round_robin.py
PYTHON=/Users/naoki/Desktop/develop/pokemon_tgc_agent/pokemon-tcg-agent/.venv/bin/python


## 設定


In [5]:
GROUP_A_ROOT = Path(
    "/Users/naoki/Desktop/develop/pokemon_tgc_agent/"
    "pokemon-tcg-agent/agents/16model_pretrained_source"
)

GROUP_B_ROOT = Path(
    "/Users/naoki/Desktop/develop/pokemon_tgc_agent/"
    "pokemon-tcg-agent/agents/16model_pretrained2"
)

GROUP_A_LABEL = (
    GROUP_A_ROOT.name
    or "group_a"
)
GROUP_B_LABEL = (
    GROUP_B_ROOT.name
    or "group_b"
)

CPU_THREADS = os.cpu_count() or 1

GAMES_PER_PAIRING = 20
DEVICE = "mps"
WORKERS = max(1, CPU_THREADS - 1)

# run_matches_round_robin.py に渡す「大会全体の同時レーン数」
LANES_PER_WORKER = 400
LANES = WORKERS * LANES_PER_WORKER

BATCH_SIZE = 256
SEARCH_COUNT = 10
MAX_TURNS = 100
MAX_SELECTIONS = 500

DRY_RUN = False

print(
    f"A: {GROUP_A_LABEL} -> "
    f"{GROUP_A_ROOT}"
)
print(
    f"B: {GROUP_B_LABEL} -> "
    f"{GROUP_B_ROOT}"
)
print(
    f"workers={WORKERS}, lanes={LANES}"
)


A: 16model_pretrained_source -> /Users/naoki/Desktop/develop/pokemon_tgc_agent/pokemon-tcg-agent/agents/16model_pretrained_source
B: 16model_pretrained2 -> /Users/naoki/Desktop/develop/pokemon_tgc_agent/pokemon-tcg-agent/agents/16model_pretrained2
workers=9, lanes=3600


## 32agentを読み込む


In [6]:
CLUSTERS = [
    f"cluster_{index:02d}"
    for index in range(16)
]


def load_group_agents(
    root: Path,
    prefix: str,
) -> dict[str, Path]:

    root = (
        root
        .expanduser()
        .resolve()
    )

    if not root.is_dir():
        raise FileNotFoundError(
            f"フォルダがありません: {root}"
        )

    agents: dict[str, Path] = {}

    for cluster in CLUSTERS:
        src = (
            root
            / cluster
            / "src"
        )

        main = src / "main.py"
        deck = src / "deck.csv"
        model = src / "model.pth"

        for required in (
            main,
            deck,
            model,
        ):
            if not required.is_file():
                raise FileNotFoundError(
                    f"必要なagentファイルがありません: "
                    f"{required}"
                )

        agents[
            f"{prefix}_{cluster}"
        ] = main.resolve()

    return agents


GROUP_A_AGENTS = load_group_agents(
    GROUP_A_ROOT,
    "groupA",
)

GROUP_B_AGENTS = load_group_agents(
    GROUP_B_ROOT,
    "groupB",
)

AGENTS = {
    **GROUP_A_AGENTS,
    **GROUP_B_AGENTS,
}

if len(AGENTS) != 32:
    raise RuntimeError(
        f"agent数が32ではありません: "
        f"{len(AGENTS)}"
    )

display(
    pd.DataFrame(
        [
            {
                "name": name,
                "main.py": str(path),
            }
            for name, path
            in AGENTS.items()
        ]
    )
)


,name,main.py
0,groupA_cluster_00,/Users/naoki/Desktop/develop/pokemon_tgc_agent...
1,groupA_cluster_01,/Users/naoki/Desktop/develop/pokemon_tgc_agent...
2,groupA_cluster_02,/Users/naoki/Desktop/develop/pokemon_tgc_agent...
3,groupA_cluster_03,/Users/naoki/Desktop/develop/pokemon_tgc_agent...
4,groupA_cluster_04,/Users/naoki/Desktop/develop/pokemon_tgc_agent...
5,groupA_cluster_05,/Users/naoki/Desktop/develop/pokemon_tgc_agent...
6,groupA_cluster_06,/Users/naoki/Desktop/develop/pokemon_tgc_agent...
7,groupA_cluster_07,/Users/naoki/Desktop/develop/pokemon_tgc_agent...
8,groupA_cluster_08,/Users/naoki/Desktop/develop/pokemon_tgc_agent...
9,groupA_cluster_09,/Users/naoki/Desktop/develop/pokemon_tgc_agent...


## 対戦実行


In [7]:
def is_cross_group(
    name0: str,
    name1: str,
) -> bool:
    return (
        name0.startswith("groupA_")
        != name1.startswith("groupA_")
    )


names = list(AGENTS)

pairings = list(
    itertools.combinations(
        names,
        2,
    )
)

counts = [
    (
        GAMES_PER_PAIRING
        if is_cross_group(
            name0,
            name1,
        )
        else 0
    )
    for name0, name1 in pairings
]

cross_pairings = sum(
    count > 0
    for count in counts
)

total_games = sum(counts)

if cross_pairings != 256:
    raise RuntimeError(
        f"cross-group pairing数が"
        f"256ではありません: "
        f"{cross_pairings}"
    )

if total_games != (
    256 * GAMES_PER_PAIRING
):
    raise RuntimeError(
        f"総試合数が不正です: "
        f"{total_games}"
    )


command = [
    str(PYTHON),
    str(RUNNER),
]

# parallel_selfplay_training.py と同じ渡し方
for name, main_path in AGENTS.items():
    command.extend(
        [
            "--agent",
            f"{name}={main_path}",
        ]
    )

command.extend(
    [
        "--backend",
        "worker-batched",

        "--device",
        DEVICE,

        "--workers",
        str(WORKERS),

        "--lanes",
        str(LANES),

        "--batch-size",
        str(BATCH_SIZE),

        "--search-count",
        str(SEARCH_COUNT),

        "--max-turns",
        str(MAX_TURNS),

        "--max-selections",
        str(MAX_SELECTIONS),

        "--games-per-pairing-json",
        json.dumps(
            counts,
            separators=(",", ":"),
        ),

        "--no-self",
        "--quiet",
        "--save-json",
    ]
)

print(
    f"cross-group pairings="
    f"{cross_pairings}"
)
print(
    f"games/pairing="
    f"{GAMES_PER_PAIRING}"
)
print(
    f"total games="
    f"{total_games}"
)

if DRY_RUN:
    print(
        shlex.join(command)
    )
    RESULT_PATH = None

else:
    existing_results = set(
        RESULTS_DIR.glob(
            "tournament_*.json"
        )
    )

    subprocess.run(
        command,
        cwd=ROOT,
        check=True,
    )

    new_results = (
        set(
            RESULTS_DIR.glob(
                "tournament_*.json"
            )
        )
        - existing_results
    )

    if not new_results:
        raise RuntimeError(
            "大会結果JSONが"
            "見つかりません。"
        )

    RESULT_PATH = max(
        new_results,
        key=lambda path: (
            path.stat().st_mtime_ns
        ),
    )

    print(
        f"result: {RESULT_PATH}"
    )


cross-group pairings=256
games/pairing=20
total games=5120
=== エージェント確認 ===
  groupA_cluster_00: /Users/naoki/Desktop/develop/pokemon_tgc_agent/pokemon-tcg-agent/agents/16model_pretrained_source/cluster_00/src/main.py (deck: /Users/naoki/Desktop/develop/pokemon_tgc_agent/pokemon-tcg-agent/agents/16model_pretrained_source/cluster_00/src/deck.csv)
  groupA_cluster_01: /Users/naoki/Desktop/develop/pokemon_tgc_agent/pokemon-tcg-agent/agents/16model_pretrained_source/cluster_01/src/main.py (deck: /Users/naoki/Desktop/develop/pokemon_tgc_agent/pokemon-tcg-agent/agents/16model_pretrained_source/cluster_01/src/deck.csv)
  groupA_cluster_02: /Users/naoki/Desktop/develop/pokemon_tgc_agent/pokemon-tcg-agent/agents/16model_pretrained_source/cluster_02/src/main.py (deck: /Users/naoki/Desktop/develop/pokemon_tgc_agent/pokemon-tcg-agent/agents/16model_pretrained_source/cluster_02/src/deck.csv)
  groupA_cluster_03: /Users/naoki/Desktop/develop/pokemon_tgc_agent/pokemon-tcg-agent/agents/16model_pretrai

## 結果集計


In [8]:
def summarize(
    result_path: Path,
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
]:

    payload = json.loads(
        result_path.read_text(
            encoding="utf-8"
        )
    )

    a_wins = 0
    b_wins = 0
    draws = 0
    unresolved = 0

    rows = []

    for (
        matchup,
        result,
    ) in payload[
        "head_to_head"
    ].items():

        name0, name1 = (
            matchup.split(
                "_vs_",
                1,
            )
        )

        if not is_cross_group(
            name0,
            name1,
        ):
            continue

        if name0.startswith(
            "groupA_"
        ):
            a_name = name0
            b_name = name1
            aw = result[
                "name0_wins"
            ]
            bw = result[
                "name1_wins"
            ]
        else:
            a_name = name1
            b_name = name0
            aw = result[
                "name1_wins"
            ]
            bw = result[
                "name0_wins"
            ]

        a_wins += aw
        b_wins += bw
        draws += result["draws"]
        unresolved += result[
            "unresolved"
        ]

        decided = aw + bw

        rows.append(
            {
                "A": (
                    a_name.replace(
                        "groupA_",
                        "",
                    )
                ),
                "B": (
                    b_name.replace(
                        "groupB_",
                        "",
                    )
                ),
                "A_wins": aw,
                "B_wins": bw,
                "draws": result[
                    "draws"
                ],
                "unresolved": result[
                    "unresolved"
                ],
                "A_decided_win_rate": (
                    aw / decided
                    if decided
                    else 0.0
                ),
            }
        )

    total = (
        a_wins
        + b_wins
        + draws
        + unresolved
    )

    decided = (
        a_wins
        + b_wins
    )

    summary = pd.DataFrame(
        [
            {
                "group": GROUP_A_LABEL,
                "wins": a_wins,
                "total_win_rate": (
                    a_wins / total
                    if total
                    else 0.0
                ),
                "decided_win_rate": (
                    a_wins / decided
                    if decided
                    else 0.0
                ),
            },
            {
                "group": GROUP_B_LABEL,
                "wins": b_wins,
                "total_win_rate": (
                    b_wins / total
                    if total
                    else 0.0
                ),
                "decided_win_rate": (
                    b_wins / decided
                    if decided
                    else 0.0
                ),
            },
        ]
    )

    matchups = (
        pd.DataFrame(rows)
        .sort_values(
            [
                "A",
                "B",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    print(
        "\n=== 合算 ==="
    )
    print(
        f"{GROUP_A_LABEL}: "
        f"{a_wins}勝 "
        f"(決着勝率 "
        f"{100 * a_wins / decided if decided else 0:.2f}%)"
    )
    print(
        f"{GROUP_B_LABEL}: "
        f"{b_wins}勝 "
        f"(決着勝率 "
        f"{100 * b_wins / decided if decided else 0:.2f}%)"
    )
    print(
        f"引き分け={draws}, "
        f"未決着={unresolved}, "
        f"合計={total}"
    )

    return (
        summary,
        matchups,
    )


if RESULT_PATH is None:
    print(
        "DRY_RUN=True のため"
        "結果はありません。"
    )
else:
    SUMMARY, MATCHUPS = summarize(
        RESULT_PATH
    )

    display(
        SUMMARY.style.format(
            {
                "total_win_rate": "{:.2%}",
                "decided_win_rate": "{:.2%}",
            }
        )
    )

    display(
        MATCHUPS.style.format(
            {
                "A_decided_win_rate": "{:.2%}",
            }
        )
    )



=== 合算 ===
16model_pretrained_source: 2499勝 (決着勝率 49.22%)
16model_pretrained2: 2578勝 (決着勝率 50.78%)
引き分け=43, 未決着=0, 合計=5120


,group,wins,total_win_rate,decided_win_rate
0,16model_pretrained_source,2499,48.81%,49.22%
1,16model_pretrained2,2578,50.35%,50.78%


,A,B,A_wins,B_wins,draws,unresolved,A_decided_win_rate
0,cluster_00,cluster_00,13,7,0,0,65.00%
1,cluster_00,cluster_01,7,13,0,0,35.00%
2,cluster_00,cluster_02,4,16,0,0,20.00%
3,cluster_00,cluster_03,11,9,0,0,55.00%
4,cluster_00,cluster_04,10,10,0,0,50.00%
5,cluster_00,cluster_05,6,14,0,0,30.00%
6,cluster_00,cluster_06,13,7,0,0,65.00%
7,cluster_00,cluster_07,15,5,0,0,75.00%
8,cluster_00,cluster_08,8,12,0,0,40.00%
9,cluster_00,cluster_09,17,3,0,0,85.00%
